# Laboratory customer support RAG — EDA
Using CRISP-DM Methodology, the objective of the project is to model whether a customer enquiry and the relevant retrieved evidence is sufficient to ANSWER or ESCALATE. We shall start by exploring information supply first, demand second, and retrieved evidence last

In [ ]:
from pathlib import Path
import json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display
SEED = 42
ROOT = Path.cwd()
# Run from project root, or its notebooks subdirectory.
if ROOT.name == 'notebooks': ROOT = ROOT.parent
def locate(*paths):
    for p in paths:
        p = ROOT / p
        if p.exists(): return p
    raise FileNotFoundError(paths)
def blank(s): return s.isna() | s.astype(str).str.strip().eq('')
def bar(s,title):
    s.plot.bar(figsize=(10,4),title=title); plt.ylabel('Count'); plt.tight_layout(); plt.show()
def hist(s,title):
    s.dropna().plot.hist(bins=30,figsize=(8,4),title=title); plt.xlabel(s.name or 'Value'); plt.tight_layout(); plt.show()
def heat(t,title):
    fig,ax=plt.subplots(figsize=(10,6)); im=ax.imshow(t.to_numpy(),aspect='auto',cmap='Blues')
    ax.set_xticks(range(len(t.columns)),t.columns,rotation=45,ha='right'); ax.set_yticks(range(len(t.index)),t.index)
    fig.colorbar(im); ax.set_title(title); plt.tight_layout(); plt.show()

def overview(x):
    print("Shape:",x.shape,"Memory bytes:",x.memory_usage(deep=True).sum())
    display(pd.DataFrame({"dtype":x.dtypes.astype(str),"unique":x.nunique(),"blank":x.apply(lambda s:blank(s).sum())}))
def norm(s): return s.str.casefold().str.replace(r"\s+"," ",regex=True).str.strip()

## 2. Laboratory catalogue
### 2.1 Basic exploration
Unit: test catalogue entry. Identifiers are administrative; names and synonyms support retrieval, and service fields provide evidence.

In [ ]:
catalogue_paths = sorted((ROOT/'data/knowledge_base').rglob('catalogue.csv'))
CAT = locate('catalogue(1).csv','catalogue.csv', *[p.relative_to(ROOT) for p in catalogue_paths if '20261116T084302Z' in str(p)])

c = pd.read_csv(CAT,keep_default_na=False,dtype={"code":str})
assert len(c)==2143
overview(c)

### 2.2 Quality
Measure field presence, not clinical correctness. Duplicate names may denote distinct services. Zero prices require review.

In [ ]:
fields=['synonyms','specimenReqQuantity','instructions','description','testDetails','reportDays','rate']
completeness=100*(1-c[fields].apply(blank).mean())
display(completeness.rename('percent populated').to_frame())
completeness.plot.barh(title='Catalogue field completeness (%)',xlim=(0,100)); plt.tight_layout(); plt.show()
print('Duplicate codes:',c.code.duplicated().sum(),'duplicate names:',c.name.str.strip().str.casefold().duplicated().sum())

price=pd.to_numeric(c.rate,errors='coerce')
print('Non-numeric prices:',price.isna().sum(),'non-positive prices:',price.le(0).sum())
display(c.reportDate.value_counts().to_frame('count'))

print("Exact duplicate rows:",c.duplicated().sum())

### 2.3 Univariate
Explore service prices, reported turnaround categories and text lengths. No unsupported conversion of working days into calendar days.

In [ ]:
display(price.describe())
hist(price.rename('Catalogue price (KES)'),'Price distribution')
price.plot.box(); plt.ylabel('KES'); plt.show()
for f in ['reportDate','specimenReqQuantity','isSpecial','testDays']:
    bar(c.loc[~blank(c[f]),f].value_counts().head(15),f+' — top observed values')
for f in ['name','description','instructions','synonyms']:
    hist(c.loc[~blank(c[f]),f].astype(str).str.len().rename(f+' characters'),f+' length')
c['populated_fields']= (~c[fields].apply(blank)).sum(axis=1)
bar(c.populated_fields.value_counts().sort_index(),'Populated catalogue fields per test')


### 2.4 Bivariate
Compare service characteristics using observed special-test flags, not invented departments.

In [ ]:
display(c.assign(price=price).groupby('isSpecial').price.describe())
c.assign(price=price).boxplot(column='price',by='isSpecial'); plt.suptitle(''); plt.show()
display(c.groupby('isSpecial')[fields].agg(lambda s:(~blank(s)).mean()))

for f in ["description","instructions","name","synonyms"]: c[f+"_length"]=c[f].astype(str).str.len()
print("Description/instruction length Spearman:",c.description_length.corr(c.instructions_length,method="spearman"))
display(pd.crosstab(blank(c.specimenReqQuantity),blank(c.instructions)))

### 2.5 Multivariate
Co-occurring missing fields indicate where evidence gaps may cluster.

In [ ]:
missing=c[fields].apply(blank).astype(int)
heat(missing.T@missing,'Catalogue missing-field co-occurrence')
heat(c[["description_length","instructions_length","name_length","synonyms_length","populated_fields"]].assign(price=price).corr(method="spearman"),"Catalogue feature correlations")

### 2.6 Findings and implications

In [ ]:
print("Least populated field:",completeness.idxmin(),round(completeness.min(),2),"%")
print("Median price:",price.median())

Implication: field presence alone cannot establish evidence sufficiency; retain missingness and retrieval distinctions.

## 3. Customer enquiries
### 3.1 Basic exploration
Unit: privacy-approved enquiry. Source identifies authentic versus generated text. Privacy flags are administrative; approval is not independent verification of de-identification.

In [ ]:
Q = locate("data/private/privacy_review_sanitized.xlsx","privacy_review_sanitized.xlsx")
q=pd.read_excel(Q,keep_default_na=False,dtype={"record_id":str})
overview(q)
assert len(q)==2611 and q.record_id.is_unique

### 3.2 Quality
Check blanks, normalized repetitions, approval and source counts. Normalized equality does not capture every semantic duplicate.

In [ ]:
print("Blank questions:",blank(q.deidentified_question).sum())
print("Normalized duplicate rows:",norm(q.deidentified_question).duplicated().sum())
display(q.privacy_approved.value_counts())
q["normalized_duplicate"]=norm(q.deidentified_question).duplicated(keep=False)
display(pd.crosstab(q.source_type,q.normalized_duplicate))

### 3.3 Univariate
Length and frequent phrases describe language and input complexity. Frequent phrases are descriptive, not fitted model vocabulary.

In [ ]:
q['word_count']=q.deidentified_question.str.split().str.len()
q['char_count']=q.deidentified_question.str.len()
for f in ['word_count','char_count']:
    display(q.groupby('source_type')[f].describe()); hist(q[f],f)
bar(q.source_type.value_counts(),'Eligible enquiry source')
# Aggregate phrases only; suppress rare phrases and do not display examples of raw enquiries.
def ngrams(texts,n):
    counts=Counter()
    for text in texts:
        tokens=re.findall(r'[a-zA-Z]+',text.lower())
        counts.update(zip(*(tokens[i:] for i in range(n))))
    return pd.Series({' '.join(k):v for k,v in counts.most_common(20) if v>=5},dtype=float)
# Optional: review aggregate vocabulary locally for residual personal identifiers before sharing.
for source,g in q.groupby('source_type'):
    for n in (1,2,3):
        bar(ngrams(g.deidentified_question,n),f'{source}: common {n}-grams')


### 3.4 Bivariate
Compare language length across sources and examine redundancy between length measures.

In [ ]:
display(q.groupby('source_type')[['word_count','char_count']].median())
q.boxplot(column='word_count',by='source_type',showfliers=False); plt.suptitle(''); plt.show()
print("Word/character Spearman:",q.word_count.corr(q.char_count,method="spearman"))
for source,gq in q.groupby("source_type"):
    print(source,"distinct tokens:",len(set(re.findall(r"[a-zA-Z]+"," ".join(gq.deidentified_question).lower()))))

### 3.5 Multivariate
Combine source, length and repetition measures to investigate synthetic-data differences.

In [ ]:
q["lexical_diversity"]=q.deidentified_question.map(lambda t: len(set(t.lower().split()))/max(1,len(t.split())))
display(q.groupby(["source_type","normalized_duplicate"])[["word_count","char_count","lexical_diversity"]].agg(["median","count"]))
heat(q[["word_count","char_count","lexical_diversity"]].corr(method="spearman"),"Text characteristic correlations")

### 3.6 Findings and implications

In [ ]:
display(q.groupby("source_type")[["word_count","char_count","lexical_diversity"]].median())


Implication: evaluate augmentation on real validation enquiries; length and lexical differences may permit shortcut learning.

## 4. Final annotated dataset
### 4.1 Basic exploration
Unit: annotated enquiry with extracted requests and retrieved candidates. `target` encodes human decision (ANSWER=1). IDs, batch, source and KB version are audit variables, not predictors.

In [ ]:
D=locate("data/private/master_annotated_dataset.csv","master_annotated_dataset.csv")
d=pd.read_csv(D,keep_default_na=False,dtype={"record_id":str,"group_id":str})
assert d.shape==(1217,24) and not {"partition","original_partition"}&set(d.columns)
overview(d)

### 4.2 Quality
Verify labels, links, structured evidence and group integrity. Missing label reasons restrict auditability.

In [ ]:
print('Group count:',d.group_id.nunique(),'multi-record groups:',(d.group_id.value_counts()>1).sum())
assert q.record_id.is_unique and d.record_id.is_unique
assert d.record_id.isin(q.record_id).all()
assert d.decision_label.isin(['ANSWER','ESCALATE']).all()
assert d.target.eq(d.decision_label.map({'ANSWER':1,'ESCALATE':0})).all()
print('Missing annotation reasons:',blank(d.label_reason).sum())
print('Extraction errors:',(~blank(d.extraction_error)).sum())
print('KB version:',d.knowledge_base_version.unique())
linked=d.merge(q[['record_id','deidentified_question','source_type']],on='record_id',suffixes=('','_eligible'),validate='one_to_one')
assert linked.question.str.strip().eq(linked.deidentified_question.str.strip()).all()
assert linked.source_type.eq(linked.source_type_eligible).all()

assert not blank(d.group_id).any()
ex=d.extraction_json.map(json.loads)
ev=d.candidate_evidence_json.map(json.loads)
assert ev.map(len).eq(d.candidate_count).all()
assert ex.map(lambda x:isinstance(x,dict)).all()

print("Normalized duplicate questions:",norm(d.question).duplicated().sum())

### 4.3 Univariate
Multi-label intent counts can exceed the number of enquiries. Target imbalance is described without selecting predictors.

In [ ]:
d['intent_count']=ex.map(lambda x:len(x['intent_group']))
d['test_count']=ex.map(lambda x:len(x['requested_tests']))
d['field_count']=ex.map(lambda x:sum(len(r['requested_fields']) for r in x['requested_tests']))
intents=pd.DataFrame([{k:1 for k in x['intent_group']} for x in ex]).fillna(0).astype(int)
requests=pd.DataFrame([{k:1 for r in x['requested_tests'] for k in r['requested_fields']} for x in ex]).fillna(0).astype(int)
for s,t in [(intents.sum().sort_values(ascending=False),'Multi-label intent frequencies'),(requests.sum().sort_values(ascending=False),'Requested field frequencies'),(d.decision_label.value_counts(),'Target imbalance'),(d.retrieval_status.value_counts(),'Retrieval status')]:
    bar(s,t)
for f in ['intent_count','test_count','field_count','candidate_count','context_dependent','ambiguity_flag']:
    bar(d[f].value_counts().sort_index(),f)
display(pd.crosstab(d.source_type,d.decision_label,margins=True))


### 4.4 Bivariate
Compare retrieval behaviour and extraction complexity without using target relationships to select features.

In [ ]:
display(intents.groupby(d.source_type).mean().T)
heat(intents.groupby(d.source_type).mean().T,'Intent proportions by source')
display(pd.crosstab(d.context_dependent,d.ambiguity_flag))
display(d[['intent_count','test_count','field_count','candidate_count']].corr(method='spearman'))
display(pd.crosstab(d.source_type,d.retrieval_status,normalize="index"))
display(intents.groupby(d.retrieval_status).sum().T)
display(d.groupby("test_count").candidate_count.describe())

### 4.5 Multivariate
Inspect joint request complexity, intent co-occurrence and ambiguity.

In [ ]:
heat(intents.T@intents,'Intent co-occurrence counts')
heat(d[['intent_count','test_count','field_count','candidate_count']].corr(method='spearman'),'Spearman feature correlations')
display(d.groupby(['source_type','context_dependent','ambiguity_flag']).size().rename('count').to_frame())
display(pd.crosstab([d.test_count,d.field_count],d.retrieval_status))

### 4.6 Findings and implications

In [ ]:
print('Real ANSWER proportion:',d.loc[d.source_type.eq('Real'),'target'].mean())
print('Synthetic ANSWER proportion:',d.loc[d.source_type.eq('Synthetic'),'target'].mean())
print('Annotated fraction:',len(d)/len(q))

Implication: assess class-specific metrics and compare real-only with augmented training."

## 5. Cross-dataset analysis
Measure annotation coverage, request demand and candidate field presence. Presence in any candidate is an upper-bound proxy, not proof of correct matching. Requested test–field pairs are the denominator.

In [ ]:
coverage=pd.concat([q.source_type.value_counts(),d.source_type.value_counts()],axis=1,keys=["eligible","annotated"])
coverage["coverage_pct"]=100*coverage.annotated/coverage.eligible
display(coverage)
mapping={'price':'price','tat':'tat','specimen':'specimen','preparation':'instructions','test_information':'description','catalogue_listing':'test_code'}
catalogue_mapping={'price':'rate','tat':'reportDays','specimen':'specimenReqQuantity','preparation':'instructions'}
def present(v): return v is not None and str(v).strip().lower() not in ('','nan','none','null','[]','{}')
def has_field(candidate,f):
    if f=='test_information': return present(candidate.get('description')) or present(candidate.get('test_details'))
    return present(candidate.get(mapping[f]))
pairs=[]; candidate_rows=[]
for i,(x,cs) in enumerate(zip(ex,ev)):
    for r in x['requested_tests']:
        matched=[a for a in cs if a['search_query']==r['test_name']]
        for f in r['requested_fields']:
            pairs.append({'row':i,'source':d.source_type.iloc[i],'test':r['test_name'],'field':f,'candidate_found':bool(matched),'field_present':any(has_field(a,f) for a in matched)})
    for a in cs:
        candidate_rows.append({'row':i,'method':a['retrieval_method'],'score':a['retrieval_score'],'rank':a['rank']})
p=pd.DataFrame(pairs); candidates=pd.DataFrame(candidate_rows)
summary=p.groupby('field').agg(demand=('field','size'),candidate_coverage=('candidate_found','mean'),field_coverage=('field_present','mean'))
summary['catalogue_completeness']=[(~blank(c[catalogue_mapping[f]])).mean() if f in catalogue_mapping else ((~blank(c.description))|(~blank(c.testDetails))).mean() if f=='test_information' else np.nan for f in summary.index]
display(summary)
display(p.groupby(['source','field']).field_present.agg(['size','mean']))
display(p.loc[p.source.eq('Real'),'test'].value_counts().head(20))
display(p.loc[p.source.eq('Real') & ~p.candidate_found,'test'].value_counts().head(20))
display(candidates.groupby('method').score.describe())
for method,g in candidates.groupby('method'):
    hist(pd.to_numeric(g.score,errors='coerce').rename('Retrieval score'),method)

display(p.groupby("row").field_present.all().value_counts())

## 6. Overall EDA synthesis
The following observed summaries support decisions; associations are descriptive, not causal. Selected annotated records need not represent natural deployment prevalence.

In [ ]:
display(summary)
display(pd.DataFrame({"Finding":["Lowest catalogue field population","Annotation coverage","Real ANSWER prevalence","Synthetic ANSWER prevalence","Missing annotation reasons"],"Observed":[f"{completeness.idxmin()}: {completeness.min():.1f}%",f"{len(d)/len(q):.1%}",f"{d.loc[d.source_type.eq('Real'),'target'].mean():.1%}",f"{d.loc[d.source_type.eq('Synthetic'),'target'].mean():.1%}",str(blank(d.label_reason).sum())],"Decision":["Preserve evidence missingness","Document selection limitations","Use class-specific metrics","Compare augmentation on real folds","Document auditability limitation"]}))

## 7. Data partitioning — CRISP-DM data preparation
Freeze 25 real ANSWER and 175 real ESCALATE records using existing groups and seed 42. Random-cost optimization is not uniform random sampling. Synthetic overlapping groups are quarantined. The enriched test distribution is not a deployment prevalence estimate. Existing groups cannot guarantee semantic independence.

In [ ]:
from scipy.optimize import milp, Bounds, LinearConstraint
real=d[d.source_type.eq('Real')]
syn=d[d.source_type.eq('Synthetic')]
g=pd.crosstab(real.group_id,real.decision_label).reindex(columns=['ANSWER','ESCALATE'],fill_value=0).sort_index()
A=g.to_numpy(dtype=float).T
n=len(g); target=np.array([25,175])
cost=np.random.default_rng(SEED).random(n)+g.index.isin(set(syn.group_id)).astype(int)*(n+1)
result=milp(c=cost,integrality=np.ones(n),bounds=Bounds(np.zeros(n),np.ones(n)),constraints=LinearConstraint(A,target,target),options={'time_limit':60})
assert result.x is not None, result.message
selected=np.rint(result.x).astype(int)
assert np.array_equal(A@selected,target)
protected=set(g.index[selected==1])
final=pd.read_csv(D,keep_default_na=False,dtype={'record_id':str,'group_id':str})
final['partition']='development'
final.loc[final.group_id.isin(protected)&final.source_type.eq('Real'),'partition']='protected'
final.loc[final.group_id.isin(protected)&final.source_type.eq('Synthetic'),'partition']='quarantined'
dev=final[final.partition.eq('development')]
test=final[final.partition.eq('protected')]
quarantine=final[final.partition.eq('quarantined')]
assert len(test)==200 and test.source_type.eq('Real').all()
assert not set(dev.group_id)&set(test.group_id)
assert not set(dev.group_id)&set(quarantine.group_id)
display(pd.crosstab([final.partition,final.source_type],final.decision_label,margins=True))
print(result.message)

assert test.decision_label.value_counts().to_dict()=={"ESCALATE":175,"ANSWER":25}
assert not set(dev.record_id)&set(test.record_id)

### 7.1 Export frozen partitions
Set SAVE_PARTITIONS to True when ready. Existing saved assignments are checked rather than overwritten.

In [ ]:
SAVE_PARTITIONS = True
if SAVE_PARTITIONS:
    # Explicit export; avoids overwriting a frozen partition on notebook reruns.
    OUT=ROOT/'data/private/final_dataset_v2'
    if OUT.exists():
        old=pd.read_csv(OUT/'final_partitions.csv',dtype={'record_id':str}).set_index('record_id').partition.sort_index()
        new=final.set_index('record_id').partition.sort_index()
        assert old.equals(new),'Existing frozen split differs; do not overwrite.'
        print('Existing frozen split verified.')
    else:
        OUT.mkdir(parents=True)
        for name,frame in [('complete_labelled_dataset',final),('development_dataset',dev),('protected_test_dataset',test),('quarantined_dataset',quarantine)]:
            frame.to_csv(OUT/f'{name}.csv',index=False,encoding='utf-8-sig')
        final[['record_id','group_id','source_type','decision_label','partition']].to_csv(OUT/'final_partitions.csv',index=False)
        (OUT/'partition_metadata.json').write_text(json.dumps({'seed':SEED,'test_answer':25,'test_escalate':175,'quarantined':len(quarantine),'optimizer_status':int(result.status),'knowledge_base_version':str(final.knowledge_base_version.iloc[0])},indent=2))
        print('Frozen partition saved:',OUT)
else:
    print("Preview only; enable SAVE_PARTITIONS to export.")

## Next: modelling
Proceed directly to feature engineering and model training on development data. Fit learned preprocessing within training folds and use real-only validation. Keep independent test untouched until final evaluation.